# Chapter 1 &mdash; Design as if There Were No Limits

**Concept 9 of the Chapter 1 decomposition:** *The Unboundedness Principle*

Identifiers, numbers and nesting have no a priori bound &mdash; so we model languages as <b>infinite</b> sets of finite strings.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Unboundedness-Principle/Concept-Unboundedness-Principle.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.LangDef        import *
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


There is usually no limit on identifier length, on how many declarations a line holds,
or on nesting depth. So `int main(){{{{{{{{}}}}}}}}` is a legal C program, and

    -12222244343434343.4343434343434343566689991

must be a legal number.

The rule: **even if a system imposes limits, design scanners and parsers as if it did
not.** This is why the whole book studies *infinite* sets rather than large finite ones.

## 2. Definitions

### Arbitrarily deep nesting, generated on demand

Nothing here caps the depth. The generator will happily produce depth 1000.

In [ ]:
def nested_braces(depth):
    return '{' * depth + '}' * depth

def c_program(depth):
    return 'int main()' + nested_braces(depth)

for d in [0, 1, 3, 8]:
    print("depth %-2d : %s" % (d, c_program(d)))
assert c_program(8).count("{") == c_program(8).count("}") == 8

### A long number

Length is bounded only by patience.

In [ ]:
long_number = '-1' + '2' * 20 + '.' + '4343' * 8 + '566689991'
print(long_number)
print("digits :", sum(c.isdigit() for c in long_number))
assert len(long_number) > 40

### Why a *finite* memory cannot check nesting

A DFA can count to a fixed bound, never to an arbitrary one. Here is one that checks
nesting **up to depth 3** &mdash; and silently fails beyond.

In [ ]:
bounded = md2mc('''DFA
IF : ( -> D1
D1 : ( -> D2
D2 : ( -> D3
D3 : ( -> BH
D1 : ) -> IF
D2 : ) -> D1
D3 : ) -> D2
IF : ) -> BH
BH : ( | ) -> BH
''')
print("bounded nesting checker, states :", sorted(bounded["Q"]))

## 3. Tests

The bounded checker is correct **only inside its bound**.

In [ ]:
for d in range(0, 6):
    s = '(' * d + ')' * d
    print("depth %d  %-14s accepted? %s" % (d, s, accepts_dfa(bounded, s)))
print()
print("Depth 4+ is rejected -- not because it is illegal, but because the")
print("machine ran out of states. THAT is why we refuse to build in a bound.")

Languages are therefore infinite. Here is `{0}*` approximated to length 6 &mdash; and the
reminder that the real thing never ends.

In [ ]:
approx = lstar({'0'}, 6)
print("lstar({'0'}, 6) has", len({s for s in approx}), "strings, longest =",
      max(len(s) for s in approx))
print("The real {0}* is infinite. We only ever LOOK at a finite window.")

## 4. Exercises


1. Extend `bounded` to depth 5. How many states did you add? Now argue that no
   *finite* extension is enough.
2. Compile `int main(){{{{{{{{}}}}}}}}` with a real C compiler. Now remove one
   `}` and read the error message carefully.
3. The known universe has about $10^{82}$ atoms. Why is a bound of $10^{82}$ still
   the wrong thing to build into a parser?

In [ ]:
# Your work for the exercises above.